In [ ]:
import pandas as pd
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np
import joblib

In [ ]:
df = pd.read_csv("fire_incident.csv", encoding='latin1')

In [ ]:
df['Date'] = pd.to_datetime(df['Date'], dayfirst=True, errors='coerce')
df = df.dropna(subset=['Date'])

In [ ]:
daily_incidents = df.groupby(df['Date'].dt.date).size()
daily_incidents.index = pd.to_datetime(daily_incidents.index)
daily_incidents = daily_incidents.sort_index()

print(f"Data range: {daily_incidents.index[0].date()} → {daily_incidents.index[-1].date()}")

Data range: 2020-01-02 → 2024-12-30


In [ ]:
# Keep only 2022
daily_incidents = daily_incidents[daily_incidents.index.year == 2022]

In [ ]:
# 70/15/15 split
n = len(daily_incidents)
train_end = int(n * 0.70)
val_end = int(n * 0.85)

train = daily_incidents.iloc[:train_end]
val = daily_incidents.iloc[train_end:val_end]
test = daily_incidents.iloc[val_end:]

print(f"\n70/15/15 Split → Train: {len(train)} | Val: {len(val)} | Test: {len(test)} days")


70/15/15 Split → Train: 95 | Val: 20 | Test: 21 days


In [ ]:
print("\nTraining initial model on train...")
model = ARIMA(train, order=(2,1,2))
model_fit = model.fit()




Training initial model on train...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
# Forecast test period
forecast_val_test = model_fit.forecast(steps=len(val) + len(test))
forecast_val = forecast_val_test[:len(val)]
forecast_test = forecast_val_test[len(val):]

/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: ValueWarning: No supported index is available. Prediction results will be given with an integer index beginning at `start`.
  return get_prediction_index(
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:837: FutureWarning: No supported index is available. In the next version, calling this method in a model without a supported index will result in an exception.
  return get_prediction_index(


In [ ]:
# Metrics
mae = mean_absolute_error(test, forecast_test)
rmse = np.sqrt(mean_squared_error(test, forecast_test))
mape = np.mean(np.abs((test - forecast_test) / (test + 1))) * 100

print(f"\nTest Results:")
print(f"MAE : {mae:.2f} incidents/day")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.1f}%")


Test Results:
MAE : 0.65 incidents/day
RMSE: 0.82
MAPE: nan%


/tmp/ipython-input-3092540496.py:4: RuntimeWarning: '<' not supported between instances of 'int' and 'Timestamp', sort order is undefined for incomparable objects.
  mape = np.mean(np.abs((test - forecast_test) / (test + 1))) * 100


In [ ]:
# Retrain on full data for final model
print("\nRetraining final model on train + validation...")
train_val = pd.concat([train, val])
final_model = ARIMA(train_val, order=(2,1,2))
final_fit = final_model.fit()


Retraining final model on train + validation...


/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)
/usr/local/lib/python3.12/dist-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: A date index has been provided, but it has no associated frequency information and so will be ignored when e.g. forecasting.
  self._init_dates(dates, freq)


In [ ]:
joblib.dump(final_fit, 'fire_arima_70_15_15.pkl')
print("\nModel saved as fire_arima_70_15_15.pkl")


Model saved as fire_arima_70_15_15.pkl
